# Contract Clause Classification - Model Comparison

This notebook compares zero-shot LLM against a fine-tuned classifier for contract clause classification on the CUAD dataset.

## Key Metrics
- **Precision**: How many selected clauses are relevant
- **Recall**: How many relevant clauses were selected
- **Cost per Document**: Operational cost comparison
- **Latency**: Response time comparison

## Setup

In [ ]:
# Import required packages
%load_ext autoreload
%autoreload 2

import os
import sys
import time
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Add project to path
sys.path.insert(0, str(Path.cwd()))

from config import config
from utils.llm_client import LLMClient
from utils.data_loader import load_cuad_dataset, preprocess_data, get_clause_distribution
from utils.classifier import FineTunedClassifier
from utils.metrics import calculate_metrics, aggregate_metrics

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA device: {torch.cuda.get_device_name(0)}')

## Configuration

In [ ]:
# Configuration
CONFIG = {
    'llm_provider': os.getenv('LLM_PROVIDER', 'openai'),
    'llm_model': os.getenv('LLM_MODEL', 'gpt-3.5-turbo'),
    'training_model': 'roberta-base',
    'max_samples': None,  # None = all samples
    'clause_types': config.data.clause_types,
    'quick_test': False,  # Set to True for faster testing
}

print('Configuration:')
for key, value in CONFIG.items():
    print(f'  {key}: {value}')

## Load Dataset

In [ ]:
# Load test dataset
print('Loading CUAD test dataset...')
contracts = load_cuad_dataset(
    split='test',
    max_samples=CONFIG['max_samples']
)
print(f'Loaded {len(contracts)} contracts')

# Get clause distribution
distribution = get_clause_distribution(contracts)
print('\nClause Distribution:')
print(distribution.to_string())

## Initialize Classifiers

In [ ]:
# Initialize zero-shot LLM client
print(f'Initializing Zero-Shot LLM: {CONFIG["llm_provider"]}/{CONFIG["llm_model"]}')
llm_client = LLMClient(
    provider=CONFIG['llm_provider'],
    model=CONFIG['llm_model']
)
print('Zero-Shot LLM initialized successfully')

# Initialize fine-tuned classifier
print(f"\nInitializing Fine-Tuned Classifier: {CONFIG['training_model']}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
fine_tuned = FineTunedClassifier(
    model_name=CONFIG['training_model'],
    device=device
)
print(f'Fine-Tuned Classifier initialized on {device}')

## Prepare Data for Evaluation

In [ ]:
# Organize data by clause type
texts_by_clause = {ct: [] for ct in CONFIG['clause_types']}
labels_by_clause = {ct: [] for ct in CONFIG['clause_types']}

for contract in contracts:
    for clause_type in CONFIG['clause_types']:
        text = contract.text[:512]  # Limit for efficiency
        is_present = contract.clauses.get(clause_type, False)
        texts_by_clause[clause_type].append(text)
        labels_by_clause[clause_type].append(1 if is_present else 0)

print('Data organized by clause type')
for clause_type in CONFIG['clause_types']:
    print(f'  {clause_type}: {len(texts_by_clause[clause_type])} samples')

## Evaluate Zero-Shot LLM

In [ ]:
# Evaluate Zero-Shot LLM
print('='*60)
print('Evaluating Zero-Shot LLM')
print('='*60)

zero_shot_metrics = {}
zero_shot_costs = []
zero_shot_latencies = []

for clause_type in CONFIG['clause_types']:
    print(f'\nEvaluating clause: {clause_type}')
    
    texts = texts_by_clause[clause_type]
    labels = labels_by_clause[clause_type]
    
    # Use subset for quick test
    eval_texts = texts[:50] if CONFIG['quick_test'] else texts
    eval_labels = labels[:50] if CONFIG['quick_test'] else labels
    
    predictions = []
    costs = []
    latencies = []
    
    for i, text in enumerate(eval_texts):
        start_time = time.time()
        response = llm_client.classify_single(text, clause_type)
        elapsed = (time.time() - start_time) * 1000  # Convert to ms
        
        is_present = response.text.strip().upper().startswith('YES')
        predictions.append(1 if is_present else 0)
        costs.append(response.cost_usd)
        latencies.append(elapsed)
    
    # Calculate metrics
    metrics = calculate_metrics(eval_labels, predictions, clause_type)
    zero_shot_metrics[clause_type] = metrics
    
    total_cost = sum(costs)
    zero_shot_costs.append(total_cost)
    zero_shot_latencies.extend(latencies)
    
    print(f'  Cost: ${total_cost:.6f}, Avg Latency: {np.mean(latencies):.2f} ms')

avg_zero_shot_latency = np.mean(zero_shot_latencies)
print(f'\nZero-Shot Summary:')
print(f'  Avg Latency: {avg_zero_shot_latency:.2f} ms')
print(f'  Total Cost: ${sum(zero_shot_costs):.6f}')

## Evaluate Fine-Tuned Model

In [ ]:
# Train or load fine-tuned model
print('='*60)
print('Training Fine-Tuned Model')
print('='*60)

if CONFIG['quick_test']:
    print('Quick test mode: Loading pre-trained model or skipping training')
    model_path = os.path.join(config.paths.models_dir, 'fine_tuned')
    if os.path.exists(model_path):
        print(f'Loading existing model from {model_path}')
        fine_tuned.load(model_path)
    else:
        print('No pre-trained model found. Training with small dataset...')
        train_contracts = load_cuad_dataset(split='train', max_samples=100)
        train_texts, train_labels = preprocess_data(train_contracts, CONFIG['clause_types'])
        train_dataset, val_dataset, _ = fine_tuned.prepare_data(train_texts, train_labels)
        training_result = fine_tuned.train(train_dataset, val_dataset)
        print(f'Training completed in {training_result.training_time_seconds:.2f} seconds')
else:
    # Train with full dataset
    print('Loading training dataset...')
    train_contracts = load_cuad_dataset(split='train', max_samples=500)
    print(f'Loaded {len(train_contracts)} training contracts')
    
    train_texts, train_labels = preprocess_data(train_contracts, CONFIG['clause_types'])
    
    print('Preparing datasets...')
    train_dataset, val_dataset, _ = fine_tuned.prepare_data(train_texts, train_labels)
    
    print('Starting training...')
    training_result = fine_tuned.train(train_dataset, val_dataset)
    print(f'\nTraining completed in {training_result.training_time_seconds:.2f} seconds')
    print(f'Training metrics: {training_result.metrics}')

In [ ]:
# Evaluate Fine-Tuned Model
print('='*60)
print('Evaluating Fine-Tuned Model')
print('='*60)

fine_tuned_metrics = {}
fine_tuned_costs = []
fine_tuned_latencies = []

for clause_type in CONFIG['clause_types']:
    print(f'\nEvaluating clause: {clause_type}')
    
    texts = texts_by_clause[clause_type]
    labels = labels_by_clause[clause_type]
    
    # Use subset for quick test
    eval_texts = texts[:50] if CONFIG['quick_test'] else texts
    eval_labels = labels[:50] if CONFIG['quick_test'] else labels
    
    # Predict
    predictions, probs, avg_latency = fine_tuned.predict(eval_texts)
    fine_tuned_latencies.append(avg_latency)
    
    # Calculate metrics
    metrics = calculate_metrics(eval_labels, predictions, clause_type)
    fine_tuned_metrics[clause_type] = metrics
    
    # Estimate cost (GPU time)
    estimated_cost = (avg_latency / 1000) * 0.001
    fine_tuned_costs.append(estimated_cost)
    
    print(f'  Cost: ${estimated_cost:.6f}, Avg Latency: {avg_latency:.2f} ms')

avg_fine_tuned_latency = np.mean(fine_tuned_latencies)
print(f'\nFine-Tuned Summary:')
print(f'  Avg Latency: {avg_fine_tuned_latency:.2f} ms')
print(f'  Total Cost: ${sum(fine_tuned_costs):.6f}')

## Comparison Results

In [ ]:
# Calculate cost per document
num_documents = len(contracts)
zero_shot_cost_per_doc = np.sum(zero_shot_costs) / num_documents
fine_tuned_cost_per_doc = np.sum(fine_tuned_costs) / num_documents

# Print comparison table
print('\n' + '='*80)
print('CLASSIFIER COMPARISON')
print('='*80)

print(f'\n{"Clause Type":<25} {"Method":<15} {"Precision":>10} {"Recall":>10} {"F1":>10} {"Acc":>10}')
print('-'*80)

for clause_type in CONFIG['clause_types']:
    zs = zero_shot_metrics[clause_type]
    ft = fine_tuned_metrics[clause_type]
    
    print(f'{clause_type:<25} {"Zero-Shot":<15} '
          f'{zs.precision:>10.4f} {zs.recall:>10.4f} {zs.f1:>10.4f} {zs.accuracy:>10.4f}')
    print(f'{clause_type:<25} {"Fine-Tuned":<15} '
          f'{ft.precision:>10.4f} {ft.recall:>10.4f} {ft.f1:>10.4f} {ft.accuracy:>10.4f}')

print('='*80)

In [ ]:
# Aggregate metrics
zero_shot_agg = aggregate_metrics(zero_shot_metrics)
fine_tuned_agg = aggregate_metrics(fine_tuned_metrics)

print('\n' + '-'*40)
print('SUMMARY STATISTICS')
print('-'*40)

print('\nZero-Shot LLM (Aggregate):')
for key, value in zero_shot_agg.items():
    print(f'  {key}: {value:.4f}')

print('\nFine-Tuned Model (Aggregate):')
for key, value in fine_tuned_agg.items():
    print(f'  {key}: {value:.4f}')

print('\n' + '-'*40)
print('COST & LATENCY')
print('-'*40)
print(f'\nZero-Shot LLM:')
print(f'  Cost per document: ${zero_shot_cost_per_doc:.6f}')
print(f'  Avg latency: {avg_zero_shot_latency:.2f} ms')

print(f'\nFine-Tuned Model:')
print(f'  Cost per document: ${fine_tuned_cost_per_doc:.6f}')
print(f'  Avg latency: {avg_fine_tuned_latency:.2f} ms')

## Visualizations

In [ ]:
# Create comparison plots
clause_types = CONFIG['clause_types']
metrics_list = ['precision', 'recall', 'f1', 'accuracy']

zs_data = {m: [] for m in metrics_list}
ft_data = {m: [] for m in metrics_list}

for clause_type in clause_types:
    zs = zero_shot_metrics[clause_type]
    ft = fine_tuned_metrics[clause_type]
    
    for m in metrics_list:
        zs_data[m].append(getattr(zs, m))
        ft_data[m].append(getattr(ft, m))

# Create subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

x = np.arange(len(clause_types))
width = 0.35

for idx, metric in enumerate(metrics_list):
    ax = axes[idx]
    
    zs_bars = ax.bar(x - width/2, zs_data[metric], width, 
                    label='Zero-Shot LLM', color='#2ecc71', alpha=0.8)
    ft_bars = ax.bar(x + width/2, ft_data[metric], width,
                    label='Fine-Tuned', color='#3498db', alpha=0.8)
    
    ax.set_ylabel(metric.capitalize())
    ax.set_title(f'{metric.capitalize()} Comparison')
    ax.set_xticks(x)
    ax.set_xticklabels(clause_types, rotation=45, ha='right')
    ax.legend()
    ax.set_ylim(0, 1.1)
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for bar, val in zip(zs_bars, zs_data[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
               f'{val:.2f}', ha='center', va='bottom', fontsize=8)
    for bar, val in zip(ft_bars, ft_data[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
               f'{val:.2f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(config.paths.outputs_dir, 'comparison_plot.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cost and Latency comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cost comparison
costs = [zero_shot_cost_per_doc, fine_tuned_cost_per_doc]
methods = ['Zero-Shot LLM', 'Fine-Tuned']
colors = ['#e74c3c', '#3498db']

bars1 = axes[0].bar(methods, costs, color=colors, alpha=0.8)
axes[0].set_ylabel('Cost per Document ($)')
axes[0].set_title('Cost per Document Comparison')
axes[0].grid(axis='y', alpha=0.3)

for bar, cost in zip(bars1, costs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.00001,
                f'${cost:.6f}', ha='center', va='bottom', fontsize=10)

# Latency comparison
latencies = [avg_zero_shot_latency, avg_fine_tuned_latency]

bars2 = axes[1].bar(methods, latencies, color=colors, alpha=0.8)
axes[1].set_ylabel('Latency (ms)')
axes[1].set_title('Average Latency Comparison')
axes[1].grid(axis='y', alpha=0.3)

for bar, lat in zip(bars2, latencies):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{lat:.2f} ms', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(config.paths.outputs_dir, 'cost_latency_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## Conclusion & Recommendations

In [ ]:
# Generate recommendations
print('\n' + '='*80)
print('ENGINEERING JUDGMENT: WHEN IS FINE-TUNING WORTH IT?')
print('='*80)

print('\n### Cost-Benefit Analysis\n')

print(f'**1. High Volume Scenarios (>1000 documents/day)**')
if fine_tuned_cost_per_doc < zero_shot_cost_per_doc:
    print(f'   Fine-tuned model is ${zero_shot_cost_per_doc - fine_tuned_cost_per_doc:.6f}/doc cheaper')
    print('   Predictable costs vs. variable LLM API costs')
else:
    print('   Zero-shot LLM is currently cheaper')

print(f'\n**2. Low Latency Requirements (<50ms response)**')
if avg_fine_tuned_latency < avg_zero_shot_latency:
    print(f'   Fine-tuned is {avg_zero_shot_latency - avg_fine_tuned_latency:.0f}ms faster')
    print('   No network overhead')
    print('   Consistent response times')
else:
    print('   Zero-shot LLM is faster')

print(f'\n**3. Data Privacy**')
print('   Fine-tuned: Data stays in-house')
print('   Zero-shot: Data sent to LLM provider')

print(f'\n**4. Model Control & Customization**')
print('   Fine-tuned: Full control over behavior')
print('   Zero-shot: Dependent on LLM provider updates')

print('\n' + '-'*60)
print('RECOMMENDATIONS')
print('-'*60)

print('\n**Use Zero-Shot LLM when:**')
print('  - Low volume (few docs/day)')
print('  - Quick prototype/POC needed')
print('  - Limited ML expertise')
print('  - Data privacy not critical')

print('\n**Use Fine-Tuned Model when:**')
print('  - High volume processing')
print('  - Cost optimization important')
print('  - Strict latency requirements')
print('  - Data privacy required')
print('  - Long-term deployment')

print('\n' + '='*80)

## Save Results

In [ ]:
# Save metrics to CSV
metrics_data = []
for clause_type in CONFIG['clause_types']:
    zs = zero_shot_metrics[clause_type]
    ft = fine_tuned_metrics[clause_type]
    
    metrics_data.extend([
        {'clause_type': clause_type, 'method': 'zero_shot', 'metric': 'precision', 'value': zs.precision},
        {'clause_type': clause_type, 'method': 'zero_shot', 'metric': 'recall', 'value': zs.recall},
        {'clause_type': clause_type, 'method': 'zero_shot', 'metric': 'f1', 'value': zs.f1},
        {'clause_type': clause_type, 'method': 'zero_shot', 'metric': 'accuracy', 'value': zs.accuracy},
        {'clause_type': clause_type, 'method': 'fine_tuned', 'metric': 'precision', 'value': ft.precision},
        {'clause_type': clause_type, 'method': 'fine_tuned', 'metric': 'recall', 'value': ft.recall},
        {'clause_type': clause_type, 'method': 'fine_tuned', 'metric': 'f1', 'value': ft.f1},
        {'clause_type': clause_type, 'method': 'fine_tuned', 'metric': 'accuracy', 'value': ft.accuracy},
    ])

metrics_df = pd.DataFrame(metrics_data)
metrics_df.to_csv(os.path.join(config.paths.outputs_dir, 'comparison_metrics.csv'), index=False)
print(f'Saved metrics to {config.paths.outputs_dir}/comparison_metrics.csv')

# Save summary
summary = f'''# Contract Clause Classification - Model Comparison Report

## Configuration
- LLM Provider: {CONFIG['llm_provider']}
- LLM Model: {CONFIG['llm_model']}
- Training Model: {CONFIG['training_model']}
- Quick Test: {CONFIG['quick_test']}

## Dataset
- Test Samples: {len(contracts)}
- Clause Types: {len(CONFIG['clause_types'])}

## Performance Summary

### Zero-Shot LLM (Aggregate)
'''

for key, value in zero_shot_agg.items():
    summary += f'- {key}: {value:.4f}\n'

summary += '''
### Fine-Tuned Model (Aggregate)
'''

for key, value in fine_tuned_agg.items():
    summary += f'- {key}: {value:.4f}\n'

summary += f'''\n
## Cost Analysis
- Zero-Shot LLM: ${zero_shot_cost_per_doc:.6f} per document
- Fine-Tuned Model: ${fine_tuned_cost_per_doc:.6f} per document

## Latency Analysis
- Zero-Shot LLM: {avg_zero_shot_latency:.2f} ms
- Fine-Tuned Model: {avg_fine_tuned_latency:.2f} ms
'''

with open(os.path.join(config.paths.outputs_dir, 'summary.md'), 'w') as f:
    f.write(summary)

print(f'Saved summary to {config.paths.outputs_dir}/summary.md')